**Record of a July 2026 Colab run.** This notebook keeps the original outputs, though its first cell was replaced to clone this repository, so those outputs came from the code of the time. To reproduce the result today, use [`docs/reproducing.md`](../docs/reproducing.md).

# Phase 4 — two-update GRPO/LoRA smoke test

This notebook runs the guarded training smoke test on a Colab GPU. It keeps the Phase 2 baseline notebook unchanged. Use a fresh GPU runtime.

## 1. Fetch the repository

In [ ]:
# Fetch the public repository.
import os
import subprocess
from pathlib import Path

repository_path = Path('/content/browser-control')
if not repository_path.exists():
    subprocess.run(['git', 'clone', '--branch', 'lfm-browser-control',
                    'https://github.com/DerrickKirimi/browser-control.git', str(repository_path)], check=True)
os.chdir(repository_path)


## 2. Install while preserving Colab's PyTorch

In [ ]:
from importlib.metadata import version

torch_before = version('torch')
print('PyTorch before install:', torch_before)
%pip install -q -e ".[environment,model,training]" pytest
torch_after = version('torch')
print('PyTorch after install:', torch_after)
if torch_after != torch_before:
    raise RuntimeError(f'PyTorch changed during install: {torch_before} -> {torch_after}')
!python -m playwright install --with-deps chromium

PyTorch before install: 2.11.0+cu128
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for browser-control-rl (pyproject.toml) ... done
PyTorch after install: 2.11.0+cu128
Installing dependencies...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive

In [ ]:
# Colab preinstalls an obsolete TorchAO that PEFT rejects during LoRA injection.
# This project does not use TorchAO, so remove it instead of loading mismatched native extensions.
%pip uninstall -y torchao

Found existing installation: torchao 0.17.0
Uninstalling torchao-0.17.0:
  Successfully uninstalled torchao-0.17.0


## 3. Validate the runtime and trainer contract

In [ ]:
!python scripts/check_colab_gpu.py
!python -m pytest -q tests/test_trl_contract.py

{
  "cudnn_version": 91900,
  "device": {
    "compute_capability": "7.5",
    "index": 0,
    "name": "Tesla T4",
    "supports_bfloat16": false,
    "total_memory_bytes": 15637086208,
    "training_dtype": "float16"
  },
  "torch_cuda_version": "12.8",
  "torch_version": "2.11.0+cu128"
}
...                                                                      [100%]


## 4. Run exactly two optimizer updates and reload the adapter

In [30]:
!python scripts/train_grpo_smoke.py

2026-07-21 18:45:53.824285: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  0% 0/2 [00:00<?, ?it/s]Could not estimate the number of tokens of the input, floating-point operations will not be computed
{'loss': 0.6836, 'grad_norm': 5.144209384918213, 'learning_rate': 0.0001, 'num_tokens': 623.0, 'completions/mean_length': 18.75, 'completions/min_length': 10.0, 'completions/max_length': 32.0, 'completions/clipped_ratio': 0.25, 'completions/mean_terminated_length': 14.333333969116211, 'completions/min_terminated_length': 10.0, 'completions/max_terminated_length': 21.0, 'rewards/click_test_reward/mean': 0.06301374733448029, 'rewards/click_test_reward/std': 0.034839384257793427, 'reward': 0.06301374733448029, 'reward_std': 0.034839384257793427, 'frac

## 5. Download the report and adapter artifact

In [31]:
import shutil
from pathlib import Path

from google.colab import files

output_path = Path('artifacts/phase4-smoke-nonzero')
report_path = output_path / 'smoke-report.json'
adapter_path = output_path / 'adapter'
status_path = output_path / 'run-status.json'
if not status_path.is_file():
    raise RuntimeError('No status exists for the nonzero-update smoke run.')
status = __import__('json').loads(status_path.read_text())
if status.get('status') != 'success':
    if report_path.is_file():
        files.download(str(report_path))
    raise RuntimeError(f'Training did not succeed; diagnostic report downloaded: {status}')
if not report_path.is_file() or not adapter_path.is_dir():
    raise RuntimeError('Successful status is missing its report or adapter.')
archive = shutil.make_archive(
    'phase4-smoke-nonzero-adapter', 'zip', adapter_path
)
files.download(str(report_path))
files.download(archive)

[browser download widget removed]

[browser download widget removed]

[browser download widget removed]

[browser download widget removed]